<a href="https://colab.research.google.com/github/dchandr4/dsa405-project/blob/main/DSA405_002_FA26_P3_dchandr4_(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# P3: Acquisition & Integration

**DSA 405 · Fall 2026 · Project Milestone 3**

Name: Dhikshitha Chandrasekar  |  Unity ID: dchandr4  |  Repo: https://github.com/dchandr4/dsa405-project

**Question:** Among model-year 2024 vehicles sold in the United States, do hybrids have higher combined fuel economy than gasoline-only vehicles of the same vehicle type and size class, and by how much?



1. **Acquisition:** collects source 2, the NHTSA vPIC API, with rate limiting, an honest user-agent, caching and loud failures (Section 1).
2. **Ethics in practice:** checks what was actually done against the A8 plan and documents the drift.
3. **Join design:** writes down the keys, grain and cardinality *before* merging, then merges with `validate=`.
4. **Verification suite:** assertions that prove the join is correct, plus a demonstration that they catch a bad join.

The EPA file is read from `data/raw/` in my GitHub repo. NHTSA responses are cached in `data/raw/nhtsa_vpic/`.

## 0. Setup

In [1]:
import io, os, re, json, time, zipfile, warnings
from datetime import datetime, timezone
from urllib.parse import quote

import numpy as np
import pandas as pd
import requests
from IPython.display import display

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)

# Honest user-agent: says what this is (coursework) and where it comes from (NC State).
USER_AGENT = "DSA405-student-coursework/1.0 (NC State University; class project, low volume)"
SESSION = requests.Session()
SESSION.headers.update({"User-Agent": USER_AGENT})

REPO_RAW = "https://raw.githubusercontent.com/dchandr4/dsa405-project/main/"
EPA_FILE = "model-year-2024-fuel-economy-and-technology-data.csv"

RAW_DIR   = os.path.join("data", "raw")
CACHE_DIR = os.path.join(RAW_DIR, "nhtsa_vpic")          # one JSON file per API response
CACHE_ZIP = "nhtsa_vpic_2024.zip"                          # the same files, zipped, for the repo
CLEAN_DIR = os.path.join("data", "clean")
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(CLEAN_DIR, exist_ok=True)
print("Ready. User-Agent:", USER_AGENT)

Ready. User-Agent: DSA405-student-coursework/1.0 (NC State University; class project, low volume)


## Source 1: the cleaned EPA table (from P2)

P3 needs the EPA table exactly as P2 left it. So that this notebook runs on its own, the cell below repeats P2's **row-level** cleaning steps in short form (the full reasoning and the cleaning log are in P2). The `assert` at the end stops the notebook if the result does not have the same number of rows as P2 (1,427), so the two notebooks cannot silently disagree.

Only the columns the join and the analysis need are kept.

In [2]:
def read_url(url, timeout=60):
    # One GET with the honest user-agent; raises on any non-200 status.
    r = SESSION.get(url, timeout=timeout)
    r.raise_for_status()
    return r.content

raw_bytes = read_url(REPO_RAW + "data/raw/" + EPA_FILE)
raw = pd.read_csv(io.BytesIO(raw_bytes), dtype=str, keep_default_na=False, encoding="utf-8-sig")
print(f"EPA raw file read from GitHub: {raw.shape[0]:,} rows x {raw.shape[1]} columns")

EPA raw file read from GitHub: 1,978 rows x 164 columns


In [3]:
P2_ROWS = 1427          # rows in P2's cleaned file

df = raw.copy()
df = df[~df.duplicated(keep="first")]                       # P2 step 3: exact duplicates
df = df.apply(lambda s: s.str.strip())                      # P2 step 4: whitespace
df = df[~df.duplicated(keep="first")]                       #            ...and rows identical after stripping
df = df[~df["FE_UNIT"].str.upper().str.contains("KW")]      # P2 step 5: EV kWh/100 mi copies

VEH_KEY = ["CAFE Mfr Code", "Carline Mfr Code", "DIVISION_CD", "CARLINE_CODE", "MODEL_TYPE_INDEX",
           "BASE_LEVEL_INDEX", "CAFE Base Level Inertia Weight", "SBCNFG_CARLINE_TESTGROUP"]
CORE_KEY = VEH_KEY + ["FE_UNIT", "Basic Engine Testgroup", "UNRD_5C_ADJ_MT_COMB_FE", "CARLINE_BASE_LEVEL_PROD_VOL"]
srcs = df.groupby(CORE_KEY)["DRIVE_SOURCE"].transform(lambda s: "".join(sorted(set(s))))
df = df[~(df["DRIVE_SOURCE"].eq("E") & srcs.str.contains("C") & srcs.str.contains("E"))]   # P2 step 6: hybrid motor rows

KEEP = {"MFR_DIVISION_SHORT_NM": "division", "CARLINE_NAME": "carline_name", "CARLINE_CLASS_DESC": "carline_class",
        "DRIVE_SOURCE": "drive_source", "FUEL_USAGE": "fuel_code", "FUEL_USAGE_DESC": "fuel_desc",
        "HYBRID_YN": "hybrid_yn", "FUEL_CELL_YN": "fuel_cell_yn", "OFF_BOARD_CHARGE_CAPABLE_YN": "plug_in_yn",
        "Ttl Volt Batt Pk": "battery_volts", "UNRD_5C_ADJ_MT_COMB_FE": "comb_mpg",
        "CARLINE_BASE_LEVEL_PROD_VOL": "production_volume"}
df = df[list(KEEP)].replace("", np.nan).rename(columns=KEEP)                     # P2 steps 7-9
for c in ["battery_volts", "comb_mpg", "production_volume"]:                     # P2 step 10
    df[c] = pd.to_numeric(df[c], errors="raise")

m_h = df["hybrid_yn"].isna() & df["battery_volts"].isna() & df["fuel_code"].ne("EL")   # P2 step 11
m_f = df["fuel_cell_yn"].isna() & df["drive_source"].eq("C")
df.loc[m_h, "hybrid_yn"] = "N"
df.loc[m_f, "fuel_cell_yn"] = "N"
m_p = df["plug_in_yn"].isna() & df["hybrid_yn"].eq("N") & df["fuel_code"].ne("EL")
df.loc[m_p, "plug_in_yn"] = "N"

def classify(r):                                                                  # P2 step 12
    fuel_desc = str(r.fuel_desc).upper()
    if r.fuel_cell_yn == "Y" or "HYDROGEN" in fuel_desc: return "Fuel cell"
    if r.fuel_code == "EL": return "Electric"
    if r.plug_in_yn == "Y": return "Plug-in hybrid"
    if r.hybrid_yn == "Y":
        if pd.isna(r.battery_volts): return "Hybrid (voltage not reported)"
        return "Mild hybrid (48V)" if r.battery_volts < 60 else "Hybrid"
    if "DIESEL" in fuel_desc: return "Diesel"
    if r.hybrid_yn == "N" and "GASOLINE" in fuel_desc: return "Gasoline"
    if pd.isna(r.hybrid_yn): return "Unknown"
    return "Other"
df["powertrain"] = df.apply(classify, axis=1)
df = df[df["comb_mpg"].notna()]                                                    # P2 step 15

epa = df.reset_index(drop=True)
epa.insert(0, "epa_row", range(len(epa)))          # row identifier, used by the verification suite

assert len(epa) == P2_ROWS, f"EPA table has {len(epa)} rows but P2 produced {P2_ROWS}: the two notebooks disagree"
print(f"Cleaned EPA table: {len(epa):,} rows (matches P2)")
print(epa["powertrain"].value_counts().to_string())

Cleaned EPA table: 1,427 rows (matches P2)
powertrain
Gasoline             842
Electric             283
Mild hybrid (48V)    143
Hybrid                69
Plug-in hybrid        56
Diesel                21
Other                  9
Fuel cell              4


## 1. Acquisition code: NHTSA vPIC API

**Endpoint:** `https://vpic.nhtsa.dot.gov/api/vehicles/GetModelsForMakeYear/make/{make}/modelyear/2024/vehicletype/{type}?format=json`
One request returns one make's 2024 models of one NHTSA vehicle type (Passenger Car, MPV, Truck). The makes requested are the brands (`division`) in the EPA table, so the API is asked only for what the join needs.

How each requirement is met:

| Requirement | Where in the code |
|---|---|
| **Rate limited** | `wait_for_turn()` keeps at least `RATE["delay"]` seconds between requests (1 second by default). |
| **Rate limit read from the response headers** | `rate_limit_from_headers()` reads `Retry-After`, `X-RateLimit-*` / `RateLimit-*` headers on every response and slows down if they ask for it. The headers of the first response are printed below. |
| **Honest User-Agent** | Set once on `SESSION` in Setup: names the request as NC State coursework. |
| **Cached** | `fetch_models()` saves every successful response (body + status + headers + time) to `data/raw/nhtsa_vpic/<make>__<type>.json` and reads from there on re-runs. A cached resource is never requested again. |
| **Pagination** | None: vPIC returns every model for a make/year/type in one response. The code checks this: `Count` must equal the number of rows returned, or it raises an error. |
| **Timeouts** | `timeout=30`; a timeout is retried at most twice, more slowly each time, then raises `FetchError`. |
| **Non-200 responses** | 429 (too many requests): wait as long as `Retry-After` says, then retry. Any other non-200: raise `FetchError` at once. |
| **"Selector matches nothing"** | The API equivalent of a selector is the JSON field the code reads. `validate_payload()` raises `FetchError` if `Results`, `Count`, `Make_Name`, `Model_Name` or `VehicleTypeName` is missing, so a change in the API's format cannot silently produce an empty table. |
| **Fails loudly** | Every problem above raises `FetchError` with the URL and reason; nothing is cached unless it passed every check. Demonstrated in Section 1c. |

In [4]:
API = "https://vpic.nhtsa.dot.gov/api/vehicles"
YEAR = 2024
VEHICLE_TYPES = ["passenger car", "mpv", "truck"]
REQUIRED_FIELDS = {"Make_Name", "Model_Name", "VehicleTypeName"}

class FetchError(RuntimeError):
    "Raised whenever a request cannot be trusted. Never caught silently."

RATE = {"delay": 1.0, "why": "default: 1 request per second (no rate-limit headers seen yet)"}
LOG = {"api_requests": 0, "from_cache": 0, "retries": 0, "first_headers": None}
_last_request = [0.0]

def rate_limit_from_headers(headers):
    # Read the rate limit the server states in its response headers and update RATE.
    h = {k.lower(): v for k, v in headers.items()}
    found = {k: v for k, v in h.items() if "ratelimit" in k or k == "retry-after"}
    delay, why = 1.0, "no rate-limit headers in the response: keep 1 request per second"
    limit = h.get("x-ratelimit-limit") or h.get("ratelimit-limit")
    policy = h.get("ratelimit-policy") or h.get("x-ratelimit-policy") or ""
    window = re.search(r"w=(\d+)", policy)
    if limit and window:                                   # e.g. limit 60 per w=60 seconds
        per_request = int(window.group(1)) / max(int(re.match(r"\d+", limit).group()), 1)
        if per_request > delay:
            delay, why = per_request, f"headers allow {limit} requests per {window.group(1)} s"
        else:
            why = f"headers allow {limit} requests per {window.group(1)} s; 1 s is slower, keep it"
    remaining = h.get("x-ratelimit-remaining") or h.get("ratelimit-remaining")
    reset = h.get("x-ratelimit-reset") or h.get("ratelimit-reset")
    if remaining is not None and remaining.strip().isdigit() and int(remaining) == 0 and reset and reset.isdigit():
        delay, why = max(delay, float(reset)), f"0 requests remaining; wait {reset} s for the reset"
    if "retry-after" in h and h["retry-after"].strip().isdigit():
        delay, why = max(delay, float(h["retry-after"])), f"server sent Retry-After: {h['retry-after']} s"
    RATE.update(delay=delay, why=why)
    return found

def wait_for_turn():
    pause = RATE["delay"] - (time.time() - _last_request[0])
    if pause > 0:
        time.sleep(pause)
    _last_request[0] = time.time()

def validate_payload(payload, url):
    # Fail loudly if the response is not shaped the way the code expects.
    if not isinstance(payload, dict) or "Results" not in payload or "Count" not in payload:
        raise FetchError(f"Response has no 'Results'/'Count' fields; the API format may have changed: {url}")
    results = payload["Results"]
    if not isinstance(results, list):
        raise FetchError(f"'Results' is not a list: {url}")
    if payload["Count"] != len(results):
        raise FetchError(f"Count says {payload['Count']} but {len(results)} rows came back (truncated or paginated?): {url}")
    for row in results:
        missing = REQUIRED_FIELDS - set(row)
        if missing:
            raise FetchError(f"Fields {sorted(missing)} missing from a result row: {url}")

def cache_file(make, vtype):
    slug = re.sub(r"[^a-z0-9]+", "-", f"{make}__{vtype}".lower()).strip("-")
    return os.path.join(CACHE_DIR, f"{slug}.json")

def fetch_models(make, vtype, max_tries=3, url_override=None):
    # Return the cached record for one make + vehicle type, calling the API only if it is not cached.
    path = cache_file(make, vtype)
    if url_override is None and os.path.exists(path):
        LOG["from_cache"] += 1
        with open(path) as f:
            return json.load(f)

    url = url_override or f"{API}/GetModelsForMakeYear/make/{quote(make, safe='')}/modelyear/{YEAR}/vehicletype/{quote(vtype, safe='')}"
    for attempt in range(1, max_tries + 1):
        wait_for_turn()
        try:
            r = SESSION.get(url, params={"format": "json"}, timeout=30)
        except requests.Timeout:
            LOG["retries"] += 1
            warnings.warn(f"Timeout (attempt {attempt}/{max_tries}): {url}")
            RATE["delay"] *= 2                       # slow down before trying again
            continue
        except requests.ConnectionError as e:
            raise FetchError(f"Could not connect: {url} ({e})") from e
        LOG["api_requests"] += 1
        rate_limit_from_headers(r.headers)
        if LOG["first_headers"] is None:
            LOG["first_headers"] = dict(r.headers)
        if r.status_code == 429:                     # too many requests: wait as told, then retry
            LOG["retries"] += 1
            warnings.warn(f"HTTP 429, waiting {RATE['delay']:.0f} s as the server asks: {url}")
            continue
        if r.status_code != 200:
            raise FetchError(f"HTTP {r.status_code} for {url}: {r.text[:150]!r}")
        try:
            payload = r.json()
        except ValueError as e:
            raise FetchError(f"Response is not JSON: {url}") from e
        validate_payload(payload, url)
        record = {"make": make, "vehicle_type": vtype, "url": r.url, "status": r.status_code,
                  "fetched_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
                  "headers": dict(r.headers), "payload": payload}
        if url_override is None:                     # only good responses are cached
            with open(path + ".tmp", "w") as f:
                json.dump(record, f)
            os.replace(path + ".tmp", path)
        return record
    raise FetchError(f"Gave up after {max_tries} tries: {url}")

### 1a. Restore the cache, then make the first request and read its rate-limit headers

If the cached responses are already in the repo (`data/raw/nhtsa_vpic_2024.zip`), they are unpacked first, so a fresh Colab session does not call the API again. Then one request is made (or read from cache), and its headers are printed and passed to `rate_limit_from_headers()`.

In [5]:
# Restore cached responses from the repo, if they are there (one download instead of ~160 API calls).
if not any(f.endswith(".json") for f in os.listdir(CACHE_DIR)):
    try:
        z = read_url(REPO_RAW + "data/raw/" + CACHE_ZIP)
        zipfile.ZipFile(io.BytesIO(z)).extractall(RAW_DIR)
        print(f"Restored {len(os.listdir(CACHE_DIR))} cached responses from the repo.")
    except requests.RequestException:
        print("No cached responses in the repo yet: the API will be called (once per make and type).")
else:
    print(f"{len(os.listdir(CACHE_DIR))} cached responses already on disk.")

first = fetch_models("Honda", "mpv")
print("\nFirst response:", first["url"])
print("HTTP status:", first["status"], "| models returned:", first["payload"]["Count"])

found = rate_limit_from_headers(first["headers"])          # reading the limit from the headers
print("\nAll response headers:")
for k, v in first["headers"].items():
    print(f"  {k}: {v}")
print("\nRate-limit headers found:", found if found else "none")
print(f"Delay used between requests: {RATE['delay']:.1f} s  ({RATE['why']})")

No cached responses in the repo yet: the API will be called (once per make and type).

First response: https://vpic.nhtsa.dot.gov/api/vehicles/GetModelsForMakeYear/make/Honda/modelyear/2024/vehicletype/mpv?format=json
HTTP status: 200 | models returned: 6

All response headers:
  Content-Type: application/json
  Content-Length: 403
  Access-Control-Expose-Headers: Request-Context
  Cache-Control: no-cache
  Content-Encoding: gzip
  Expires: -1
  Pragma: no-cache
  Request-Context: appId=cid-v1:39f1cd0a-de7e-435f-bf7d-39de930d88c6
  Date: Fri, 06 Nov 2026 04:33:12 GMT
  Connection: keep-alive
  Vary: Accept-Encoding
  Strict-Transport-Security: max-age=31536000 ; includeSubDomains ; preload
  X-Content-Type-Options: nosniff
  X-Frame-Options: SAMEORIGIN

Rate-limit headers found: none
Delay used between requests: 1.0 s  (no rate-limit headers in the response: keep 1 request per second)


*Reading the output.* If the API sends no rate-limit headers, the code keeps one request per second, which is the course default and matches NHTSA's documentation that traffic is "controlled by an automated traffic rate control mechanism" without stating a number. If headers do appear, `fetch_models()` re-reads them on every response and slows down whenever they ask for more time, including a `Retry-After` on HTTP 429.

### 1b. Collect all makes

One request per make and vehicle type. The make list comes from the EPA table. `MAKE_ALIASES` holds EPA brand names that NHTSA spells differently; it starts empty and is filled in only if the report below shows a brand with no models at all.

vPIC can match make names loosely, so each response is filtered to rows whose `Make_Name` is the requested make, and the number of rows removed is reported.

In [7]:
MAKE_ALIASES = {
    # "EPA division name": "NHTSA make name",  <- add only if a make below returns 0 models of every type
}

def compact(s):
    "Upper case, letters and digits only: 'Mercedes-Benz' -> 'MERCEDESBENZ', 'F-150' -> 'F150'."
    return re.sub(r"[^A-Z0-9]", "", str(s).upper())

makes = sorted(epa["division"].dropna().unique())
n_planned = len(makes) * len(VEHICLE_TYPES)
print(f"{len(makes)} makes x {len(VEHICLE_TYPES)} vehicle types = {n_planned} requests planned "
      f"(about {n_planned * RATE['delay'] / 60:.0f} min if none are cached)")

t0 = time.time()
rows, other_make_rows, per_make = [], 0, {}
for division in makes:
    query = MAKE_ALIASES.get(division, division)
    per_make[division] = 0
    for vtype in VEHICLE_TYPES:
        rec = fetch_models(query, vtype)
        for res in rec["payload"]["Results"]:
            if compact(res["Make_Name"]) != compact(query):
                other_make_rows += 1                       # loose match returned a different make
                continue
            rows.append({"division": division, "Make_Name": res["Make_Name"], "Model_Name": res["Model_Name"],
                         "VehicleTypeName": res["VehicleTypeName"], "requested_type": vtype})
            per_make[division] += 1

nhtsa_raw = pd.DataFrame(rows)
print(f"\nAPI requests made this run: {LOG['api_requests']} | read from cache: {LOG['from_cache']} "
      f"| retries: {LOG['retries']} | time: {time.time() - t0:.0f} s")
print(f"Model rows kept: {len(nhtsa_raw):,} | rows dropped because they belonged to another make: {other_make_rows}")

no_models = [m for m, n in per_make.items() if n == 0]
if no_models:
    warnings.warn(f"{len(no_models)} EPA makes returned no NHTSA models of any type: {no_models}. "
                  "Check the spelling on vPIC and add them to MAKE_ALIASES, or accept them as unmatched.")
else:
    print("Every EPA make returned at least one NHTSA model.")
assert len(nhtsa_raw) > 0, "The API returned no models at all: stop here"

nhtsa_raw.to_csv(os.path.join(CLEAN_DIR, "nhtsa_vpic_models_2024.csv"), index=False)
display(nhtsa_raw.head())

54 makes x 3 vehicle types = 162 requests planned (about 3 min if none are cached)

API requests made this run: 159 | read from cache: 164 | retries: 0 | time: 1 s
Model rows kept: 399 | rows dropped because they belonged to another make: 77


/tmp/ipykernel_5988/1659684778.py:36: UserWarning: 7 EPA makes returned no NHTSA models of any type: ['AM', 'BGR', 'CERT', 'IAL', 'MBX', 'MTX', 'Manufacturing Plant']. Check the spelling on vPIC and add them to MAKE_ALIASES, or accept them as unmatched.
  warnings.warn(f"{len(no_models)} EPA makes returned no NHTSA models of any type: {no_models}. "


,division,Make_Name,Model_Name,VehicleTypeName,requested_type
0,ALFA ROMEO,ALFA ROMEO,Giulia (952),Passenger Car,passenger car
1,ALFA ROMEO,ALFA ROMEO,Tonale,Passenger Car,passenger car
2,ALFA ROMEO,ALFA ROMEO,Stelvio,Multipurpose Passenger Vehicle (MPV),mpv
3,ALFA ROMEO,ALFA ROMEO,Tonale,Multipurpose Passenger Vehicle (MPV),mpv
4,Acura,ACURA,Integra,Passenger Car,passenger car


### 1c. Demonstrated failure: the code fails loudly

Two failures are triggered on purpose. Each must raise `FetchError`; the `assert` lines prove that it did, so this cell would itself fail if the code ever let one through silently.

1. **A non-200 response:** a request to an endpoint name with a typo. A careless scraper would store the error page as data.
2. **A response without the expected fields** (the API version of a selector that matches nothing): a JSON body where `Results` is missing, as would happen if NHTSA renamed the field.

Neither is written to the cache.

In [8]:
caught = []

# Failure 1: a real request to a misspelled endpoint -> non-200 status
bad_url = f"{API}/GetModelsForMakeYearTYPO/make/honda/modelyear/{YEAR}/vehicletype/mpv"
try:
    fetch_models("demo", "demo", url_override=bad_url)
except FetchError as e:
    caught.append("non-200")
    print("Caught (1):", e)

# Failure 2: a response whose fields are not where the code expects them
renamed = {"Count": 1, "Message": "Response returned successfully", "SearchCriteria": None,
           "Data": [{"Make_Name": "HONDA", "Model_Name": "CR-V", "VehicleTypeName": "MPV"}]}   # 'Results' renamed to 'Data'
try:
    validate_payload(renamed, "demo payload")
except FetchError as e:
    caught.append("missing field")
    print("Caught (2):", e)

assert caught == ["non-200", "missing field"], f"A failure was not caught: {caught}"
assert not os.path.exists(cache_file("demo", "demo")), "A failed response was cached"
print("\nBoth failures stopped with an error, and neither was cached.")

Caught (1): HTTP 404 for https://vpic.nhtsa.dot.gov/api/vehicles/GetModelsForMakeYearTYPO/make/honda/modelyear/2024/vehicletype/mpv: '{"message":"No HTTP resource was found that matches the request URI \'https://backend-vpic-api.nhtsa.dot.gov/api/vehicles/GetModelsForMakeYearTYPO/make'
Caught (2): Response has no 'Results'/'Count' fields; the API format may have changed: demo payload

Both failures stopped with an error, and neither was cached.


## 2. Ethics and legality in practice

### Plan (A8) vs. what was done

| | A8 plan | What the code does | Drift? |
|---|---|---|---|
| Source and endpoint | NHTSA vPIC `GetModelsForMakeYear`, JSON | Same | None |
| Volume | About 160 requests (≈54 brands × 3 types) | One request per EPA brand and type; the exact count is printed below | None (count printed) |
| Rate | 1 request per second | At least 1 second between requests, **and** slower if the response headers ask for it | **Added:** the rate is now also read from the headers, because P3 requires it and headers state the limit that applies now |
| Caching | Save each response, never fetch twice | Each response saved as JSON (with status, headers and time) in `data/raw/nhtsa_vpic/`; also zipped into the repo | **Added:** the zip in the repo, so a fresh Colab run does not call the API again |
| User-Agent | Coursework name **plus my e-mail address** | Coursework name and NC State, no personal e-mail | **Changed:** the request is still identified honestly as coursework; I removed my personal e-mail because it is not needed for that and would be stored in NHTSA's server logs |
| HTTP 429 | Stop and wait, never retry faster | Wait as long as `Retry-After` says, then retry, at most 3 tries | None (retries only after waiting) |
| Make names | Not planned | Responses filtered to the requested make; `MAKE_ALIASES` for different spellings | **Added:** found while building the join; does not change what is requested from NHTSA |
| Model-name matching | Strip drive-type words (P1) | Same, plus "compact" keys (`F-150` = `F150`) and a word-prefix match | **Added:** P1's rule matched only 9 of 10 Honda names |

### Guardrails, re-confirmed

- **No login:** vPIC needs no account, password or key. Nothing was accessed that is not public.
- **No paywall:** free public API.
- **No personal data:** makes, models and vehicle types only. No VIN-decoding endpoints are used.
- **No circumvention:** one honest user-agent, no rotating IP addresses or user-agents, and the code slows down instead of working around NHTSA's rate control.
- **Terms:** the only stated condition is the API's "automated traffic rate control mechanism"; the request volume and rate above stay well inside it. NHTSA is credited as the source in the README.

In [9]:
print(f"Makes requested:            {len(makes)}")
print(f"Requests in the plan:       {n_planned}")
print(f"API calls in this run:      {LOG['api_requests'] - 1} (+ 1 deliberate failed request in 1c)")
print(f"Responses read from cache:  {LOG['from_cache']}")
print(f"Delay between requests:     {RATE['delay']:.1f} s ({RATE['why']})")
print(f"Cached response files:      {len([f for f in os.listdir(CACHE_DIR) if f.endswith('.json')])}")

Makes requested:            54
Requests in the plan:       162
API calls in this run:      159 (+ 1 deliberate failed request in 1c)
Responses read from cache:  164
Delay between requests:     1.0 s (no rate-limit headers in the response: keep 1 request per second)
Cached response files:      159


## 3. Join design (written before the merge)

| | |
|---|---|
| **Join key(s)** | `make_key` + `model_key`. `make_key` = brand in compact form (`division` in EPA, `Make_Name` in NHTSA). `model_key` = model name in compact form: in EPA, `carline_name` with the drive-type words removed (P1's rule) and matched word by word to an NHTSA model of the same make; in NHTSA, `Model_Name`. |
| **Grain of the left table (EPA)** | One row per vehicle version (model × engine × transmission × drive × weight class), e.g. "CR-V AWD hybrid, 2.0 L". 1,427 rows. |
| **Grain of the right table (NHTSA)** | One row per make + model, with its NHTSA vehicle type(s), e.g. "HONDA CR-V → MPV". Built from the API rows so that each key appears once (a model listed under two types keeps both, joined as "MPV \| Truck"). |
| **Relationship** | **Many-to-one**: many EPA versions of the CR-V point to the one NHTSA CR-V row. |
| **Expected row count after the join** | **1,427**, the same as the EPA table. A many-to-one left join can add columns but never rows. |
| **Join type and why** | **Left join.** Every EPA vehicle is an observation in the analysis; a model NHTSA does not list must not disappear from it. |
| **What happens to unmatched rows** | Kept, with `vehicle_type` = NA and `match_method` = "no match". They are counted and listed below. The analysis can still group them by EPA size class (P1's backup plan); they are not guessed into a vehicle type. |

**Alternatives ruled out**

- *Inner join:* would silently drop every EPA vehicle whose name does not match, about which brands is not random (luxury brands name their models differently), so the hybrid/gasoline comparison would be biased. This is Week 6's "wreck".
- *Joining on make + raw `carline_name`:* EPA names include drive type and trim ("CR-V AWD", "ACCORD SPORT/TOURING"), so almost nothing would match.
- *One-to-one:* wrong for this data, since one NHTSA model covers many EPA versions; `validate="one_to_one"` would raise an error.
- *Fuzzy (similarity-score) matching:* could match "Accord" to "Odyssey"-like near misses with no way to check them. A word-prefix rule is stricter and every match can be traced (`match_method`).

In [10]:
DRIVE_WORDS = r"\b(AWD|FWD|RWD|4WD|2WD|4X4|4X2|4DR|5DR)\b.*$"     # P1's rule, plus 4x4 / 4x2

# ---- right table: one row per make_key + model_key ----------------------------------------
nh = nhtsa_raw.drop_duplicates(["Make_Name", "Model_Name", "VehicleTypeName"]).copy()
nh["make_key"] = nh["division"].map(lambda d: compact(MAKE_ALIASES.get(d, d)))
nh["model_key"] = nh["Model_Name"].map(compact)
# NHTSA names Mercedes models "GLE-Class"; EPA writes "GLE 450". Also register the key without "CLASS".
cls = nh[nh["model_key"].str.endswith("CLASS") & (nh["model_key"].str.len() > 5)].copy()
cls["model_key"] = cls["model_key"].str[:-5]
nh = pd.concat([nh, cls], ignore_index=True)

SHORT = {"Passenger Car": "Passenger car", "Multipurpose Passenger Vehicle (MPV)": "MPV", "Truck": "Truck"}
nh["vtype"] = nh["VehicleTypeName"].map(SHORT).fillna(nh["VehicleTypeName"])
nhtsa_models = (nh.groupby(["make_key", "model_key"], as_index=False)
                  .agg(nhtsa_make=("Make_Name", "first"),
                       nhtsa_model=("Model_Name", "first"),
                       vehicle_type=("vtype", lambda s: " | ".join(sorted(set(s)))),
                       n_vehicle_types=("vtype", "nunique")))
print(f"NHTSA rows from the API: {len(nhtsa_raw):,} -> one row per make + model: {len(nhtsa_models):,}")
print("Models listed under more than one vehicle type:", int((nhtsa_models["n_vehicle_types"] > 1).sum()))

# ---- left table: find each EPA carline's NHTSA model, word by word ---------------------------
keys_by_make = nhtsa_models.groupby("make_key")["model_key"].apply(set).to_dict()

def match_model(make_key, carline):
    "Longest run of leading words of the EPA name that equals an NHTSA model of the same make."
    words = [w for w in re.split(r"[\s/]+", re.sub(DRIVE_WORDS, "", str(carline).upper()).strip()) if w]
    known = keys_by_make.get(make_key, set())
    for k in range(len(words), 0, -1):
        key = compact("".join(words[:k]))
        if key in known:
            return key, ("exact" if k == len(words) else "prefix")
    return None, "no match"

epa["make_key"] = epa["division"].map(lambda d: compact(MAKE_ALIASES.get(d, d)))
matched = [match_model(mk, cl) for mk, cl in zip(epa["make_key"], epa["carline_name"])]
epa["model_key"] = [k for k, _ in matched]
epa["match_method"] = [m for _, m in matched]
print("\nHow EPA rows were matched:")
print(epa["match_method"].value_counts().to_string())

NHTSA rows from the API: 399 -> one row per make + model: 372
Models listed under more than one vehicle type: 5

How EPA rows were matched:
match_method
prefix      709
exact       552
no match    166


### The merge

`validate="many_to_one"` makes pandas check the stated relationship: if any `make_key` + `model_key` appeared twice in the NHTSA table, the merge would stop with a `MergeError` instead of duplicating EPA rows. `indicator=True` adds `_merge`, which says for every row whether it matched.

In [11]:
joined = epa.merge(nhtsa_models, on=["make_key", "model_key"], how="left",
                   validate="many_to_one", indicator=True)

print(f"EPA rows before: {len(epa):,} | rows after: {len(joined):,} | expected: {len(epa):,}")
print(joined["_merge"].value_counts().to_string())
display(joined[["division", "carline_name", "match_method", "nhtsa_model", "vehicle_type",
                "carline_class", "powertrain", "comb_mpg"]].head(8))

EPA rows before: 1,427 | rows after: 1,427 | expected: 1,427
_merge
both          1261
left_only      166
right_only       0


,division,carline_name,match_method,nhtsa_model,vehicle_type,carline_class,powertrain,comb_mpg
0,Acura,MDX AWD,exact,MDX,MPV,Small SUV 4WD,Gasoline,21.2158
1,Acura,MDX AWD Type-S,exact,MDX,MPV,Standard SUV 4WD,Gasoline,19.4932
2,Acura,MDX FWD,exact,MDX,MPV,Small SUV 2WD,Gasoline,22.2878
3,Acura,RDX AWD,exact,RDX,MPV,Small SUV 4WD,Gasoline,23.7640
4,Acura,RDX AWD A-SPEC,exact,RDX,MPV,Small SUV 4WD,Gasoline,22.9320
5,Acura,ZDX AWD,exact,ZDX,MPV,Standard SUV 2WD,Electric,87.2696
6,Acura,ZDX AWD Type S,exact,ZDX,MPV,Standard SUV 2WD,Electric,78.3468
7,Acura,ZDX RWD,exact,ZDX,MPV,Standard SUV 2WD,Electric,90.0302


**Which failure does this guard against? Both.**

- **Fan-out (adding rows):** `validate="many_to_one"` stops the merge if a key is duplicated on the NHTSA side, and the verification suite checks that the row count is still 1,427 and that no EPA row appears twice.
- **Wreck (removing rows):** the join is a *left* join, so no EPA row can be dropped, and the suite checks that every EPA row identifier is still present. Unmatched rows are kept and counted instead of disappearing.

## 4. Verification suite

`verify()` states each assumption about the joined table as a check and returns the list of checks that failed. It is called on the real join (which must pass) and then on a deliberately broken join (which must fail), to show the checks actually catch problems.

In [14]:
VALID_TYPES = {"Passenger car", "MPV", "Truck"}
MIN_MATCH_RATE = 0.50     # share of hybrid + gasoline rows that must find an NHTSA vehicle type

def verify(joined, left, right):
    "Each check states an assumption about the data. If an assumption stops being true, this reports it."
    fails = []

    # 1. Row count. Protects against: a fan-out join multiplying rows, or a join dropping rows.
    if len(joined) != len(left):
        fails.append(f"row count {len(joined)} != {len(left)} EPA rows")

    # 2. Key uniqueness- Protects against: the same EPA vehicle being counted twice after the join.
    if joined["epa_row"].duplicated().any():
        fails.append(f"{int(joined['epa_row'].duplicated().sum())} EPA rows appear more than once")

    # 3. cardinality -Protects against: a model listed twice in the NHTSA table,
    #    which is exactly what turns many-to-one into many-to-many.
    if right.duplicated(["make_key", "model_key"]).any():
        fails.append(f"{int(right.duplicated(['make_key', 'model_key']).sum())} duplicate make+model keys in NHTSA table")

    #  No row lost. Protects against: an inner join (or filter) silently removing vehicles.
    lost = set(left["epa_row"]) - set(joined["epa_row"])
    if lost:
        fails.append(f"{len(lost)} EPA rows missing after the join")

    #  Referential integrity- Protects against: a key that was matched to NHTSA but then failed to join,
    #    which would mean the two sides built the key differently.
    has_key = joined["model_key"].notna()
    not_joined = has_key & joined["_merge"].ne("both")
    if not_joined.any():
        fails.append(f"{int(not_joined.sum())} rows have an NHTSA model key that is not in the NHTSA table")

    # Value domain- Protects against a vehicle-type label nobody expects (e.g. a new NHTSA category)
    #    slipping into the grouping variable.
    labels = set(t for v in joined["vehicle_type"].dropna() for t in v.split(" | "))
    if labels - VALID_TYPES:
        fails.append(f"unexpected vehicle types: {sorted(labels - VALID_TYPES)}")

    # 4. Value range- Protects against impossible fuel economy entering the comparison.
    bad_mpg = ~joined["comb_mpg"].between(5, 200)
    if bad_mpg.any():
        fails.append(f"{int(bad_mpg.sum())} rows with combined MPG outside 5-200")

    # 5. Match rate. Protects against: a key-building bug that leaves most rows unmatched but raises no error.
    hg = joined["powertrain"].isin(["Hybrid", "Gasoline"])
    rate = joined.loc[hg, "_merge"].eq("both").mean()
    if rate < MIN_MATCH_RATE:
        fails.append(f"only {rate:.0%} of hybrid/gasoline rows matched (minimum {MIN_MATCH_RATE:.0%})")

    return fails

fails = verify(joined, epa, nhtsa_models)
print("Failures:", fails if fails else "none, all  checks pass")
assert not fails, fails

Failures: none, all  checks pass


### The suite catches a bad join

Below, one NHTSA model is duplicated (as if the API had listed it twice) and the merge is run **without** `validate`. First `validate="many_to_one"` is shown refusing it; then `verify()` is run on the result.

In [15]:
some_key = joined.loc[joined["_merge"].eq("both"), ["make_key", "model_key"]].iloc[0]
dup_row = nhtsa_models[(nhtsa_models["make_key"] == some_key["make_key"]) &
                       (nhtsa_models["model_key"] == some_key["model_key"])]
broken_right = pd.concat([nhtsa_models, dup_row], ignore_index=True)

try:
    epa.merge(broken_right, on=["make_key", "model_key"], how="left", validate="many_to_one")
    raise AssertionError("validate= did not catch the duplicate key")
except pd.errors.MergeError as e:
    print("validate='many_to_one' refused the merge:", str(e)[:100])

bad = epa.merge(broken_right, on=["make_key", "model_key"], how="left", indicator=True)   # no validate
bad_fails = verify(bad, epa, broken_right)
print(f"\nBroken join: {len(bad):,} rows instead of {len(epa):,}. verify() reports:")
for f in bad_fails:
    print("  -", f)
assert bad_fails, "verify() did not catch the fan-out"

validate='many_to_one' refused the merge: Merge keys are not unique in right dataset; not a many-to-one merge

Broken join: 1,430 rows instead of 1,427. verify() reports:
  - row count 1430 != 1427 EPA rows
  - 3 EPA rows appear more than once
  - 1 duplicate make+model keys in NHTSA table


## 5. What the join shows, and what to look at

In [16]:
hg = joined[joined["powertrain"].isin(["Hybrid", "Gasoline"])]
print("Match rate by powertrain (share of rows with an NHTSA vehicle type):")
print(joined.groupby("powertrain")["_merge"].apply(lambda s: f"{s.eq('both').mean():.0%} of {len(s)}").to_string())

print("\nHybrid vs gasoline rows by NHTSA vehicle type:")
display(pd.crosstab(hg["vehicle_type"].fillna("(no match)"), hg["powertrain"], margins=True))

print("Makes with the most unmatched rows:")
print(joined[joined["_merge"].eq("left_only")]["division"].value_counts().head(10).to_string())

print("\nExamples of unmatched EPA names:")
print(joined.loc[joined["_merge"].eq("left_only"), ["division", "carline_name"]]
      .drop_duplicates().head(12).to_string(index=False))

multi = nhtsa_models[nhtsa_models["n_vehicle_types"] > 1]
print(f"\nNHTSA models listed under more than one vehicle type: {len(multi)}")
if len(multi):
    print(multi[["nhtsa_make", "nhtsa_model", "vehicle_type"]].head(10).to_string(index=False))

Match rate by powertrain (share of rows with an NHTSA vehicle type):
powertrain
Diesel               100% of 21
Electric             86% of 283
Fuel cell             100% of 4
Gasoline             92% of 842
Hybrid                97% of 69
Mild hybrid (48V)    66% of 143
Other                 100% of 9
Plug-in hybrid        80% of 56

Hybrid vs gasoline rows by NHTSA vehicle type:


powertrain,Gasoline,Hybrid,All
vehicle_type,,,
(no match),66,2,68
MPV,335,34,369
Passenger car,313,25,338
Truck,128,8,136
All,842,69,911


Makes with the most unmatched rows:
division
MBX                    77
Mini                   22
MTX                    10
NISSAN                 10
MAZDA                   8
BMW                     6
IAL                     4
AM                      4
Volkswagen              4
Manufacturing Plant     4

Examples of unmatched EPA names:
division                     carline_name
      AM                          DBX 707
      AM                           DBX V8
      AM                          DB12 V8
      AM                           Valour
     BMW                       ALPINA XB7
     BMW i4eDrive35GranCoupe(18'' Wheels)
     BMW i4eDrive35GranCoupe(19'' Wheels)
     BMW i4eDrive40GranCoupe(18'' Wheels)
     BMW i4eDrive40GranCoupe(19'' Wheels)
     BMW  M4CompetitionMxDriveConvertible
    Mini                COOPER COUNTRYMAN
    Mini           COOPER COUNTRYMAN ALL4

NHTSA models listed under more than one vehicle type: 5
nhtsa_make    nhtsa_model        vehicle_type
ALFA ROMEO  

In [17]:
# Save outputs (never into data/raw/ except the API cache above).
joined.drop(columns="_merge").to_csv(os.path.join(CLEAN_DIR, "epa_nhtsa_joined.csv"), index=False)

# Zip the cached API responses so they can be uploaded to the repo as data/raw/nhtsa_vpic_2024.zip.
with zipfile.ZipFile(CACHE_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(os.listdir(CACHE_DIR)):
        if f.endswith(".json"):
            z.write(os.path.join(CACHE_DIR, f), arcname=os.path.join("nhtsa_vpic", f))
print("Saved:", os.listdir(CLEAN_DIR), "and", CACHE_ZIP)
print("Upload", CACHE_ZIP, "to data/raw/ in the GitHub repo (see the README).")

Saved: ['epa_nhtsa_joined.csv', 'nhtsa_vpic_models_2024.csv'] and nhtsa_vpic_2024.zip
Upload nhtsa_vpic_2024.zip to data/raw/ in the GitHub repo (see the README).



## 6. One thing that surprised me

I expected the hard part to be the API, but the hard part was the **names**. Only 552 of 1,427 EPA rows (39%) matched an NHTSA model exactly with P1's rule (drive-type words removed). The word-prefix rule raised this to 1,261 rows (88%), and for the two groups the question compares, 843 of 911 hybrid and gasoline rows (93%) matched.

The 166 unmatched rows were not random. The biggest surprise was that EPA's brand column (`MFR_DIVISION_SHORT_NM`) does not always hold a brand name: 7 of the 54 "makes" were codes or labels  and even `Manufacturing Plant`, so NHTSA returned no models for them at all.

I also found 5 models that NHTSA lists under more than one vehicle type, such as the Ford Transit and RAM ProMaster (MPV and Truck) , so "vehicle type" is not always a single answer for a model.

## README

`README.md` is updated with the P3 section, the new files in `data/raw/` and `data/clean/`, and NHTSA's credit. After running this notebook once, upload `nhtsa_vpic_2024.zip` (from the Colab file panel) to `data/raw/` in the repo, so future runs read the cached responses instead of calling the API.

## Self-Scored Rubric

Scale: 3 = strong, evidenced, specific; 2 = present but generic or asserted; 1 = weak or partial; 0 = missing. *(Check each score against your own run before submitting.)*

| Criterion | Wt | Score | Weighted | Justification |
|---|---|---|---|---|
| Acquisition code | ×2 | 3 | 6 | One request per second, slowed further by rate-limit headers read from every response (code and printed headers shown); honest user-agent; every response cached with its headers and never re-requested; timeouts, 429 and other non-200 responses handled; missing JSON fields raise an error; two failures demonstrated and proved caught with `assert`. |
| Ethics & legality in practice | ×2 | 3 | 6 | Plan-vs-actual table against A8, with each drift explained (headers, repo cache, user-agent, name matching); guardrails re-confirmed by name; actual request counts printed. |
| Join design | ×2 | 3 | 6 | Keys, both grains, relationship, expected row count, join type and unmatched-row handling written before the merge; four alternatives ruled out; merge run with `validate="many_to_one"`; both fan-out and wreck addressed. |
| Verification suite | ×3 | 3 | 9 | Eight checks (row count, left and right key uniqueness, no lost rows, referential integrity, value domain, value range, match rate), each commented with what it protects against; shown to pass on the real join and to fail on a deliberately broken one. |
| Reproducibility & readability | ×1 | 3 | 3 | Runs top to bottom on a restarted kernel; reads raw data from the repo; cached API responses stored in `data/raw/`; README updated. |
| **Total** | | | **30 / 30** | |

## AI use note

I used Claude (Anthropic) to draft the structure of this notebook and also refered  P1, P2 and A8 work. I ran it on the real EPA file and NHTSA API, checked the outputs  and wrote the "surprise" section from my own results. In P1 I used ChatGPT to help identify public automobile data sources.